<a href="https://colab.research.google.com/github/Muhozgu/chlorobiota/blob/main/chlorobiota.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers==4.51.3 accelerate bitsandbytes qwen-vl-utils pillow -q

In [2]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from PIL import Image

MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL, use_fast=False)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
).eval()
print("Ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Ready!


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("❌ No GPU detected — change runtime to GPU in Colab")

CUDA available: True
Device count: 1
PyTorch version: 2.10.0+cu128
GPU: Tesla T4
VRAM: 15.6 GB


In [7]:
def identify(image_path):
    image = Image.open(image_path).convert("RGB")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": """You are a botanist. Identify this plant:

Common name:
Family name:
Scientific name:
Description: """},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=300, do_sample=False, repetition_penalty=1.05)
    trimmed = out[0][inputs['input_ids'].shape[1]:]
    return processor.decode(trimmed, skip_special_tokens=True)

# Usage
print(identify("/content/photo.jpg"))
print(identify("/content/images.jpg"))

The plant in the picture is a Monstera deliciosa, commonly known as the Swiss Cheese Plant.

- **Common Name:** Monstera deliciosa
- **Family Name:** Araceae (Arum family)
- **Scientific Name:** Monstera deliciosa

**Description:**
Monstera deliciosa is a tropical evergreen vine native to Central and South America. It is characterized by its large, glossy, dark green leaves with distinctive splits or holes, which give it its common name "Swiss Cheese Plant." These leaves can grow quite large, often reaching up to 2 feet long. The plant is popular in both indoor and outdoor gardens due to its striking foliage and ability to thrive in various conditions. It is also known for its edible fruit, which has a sweet, custard-like flavor similar to a banana.
The plant in the picture is a **Papaya**.

- **Common Name:** Papaya
- **Family Name:** Caricaceae (Papaya Family)
- **Scientific Name:** Carica papaya

**Description:**
The Papaya, also known as the "pawpaw" or "papaw," is a tropical fruit